# K 值研究

看不同状态数下，A 股市场被切成了什么样。

逻辑都在 `experiment/k_study.py`，这个 notebook 只是调用 + 看图。


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from experiment import k_study

feats, close = k_study.load_features("ashare")
regular, ext = k_study.split_extreme(feats, "ashare")
print("特征", feats.shape, "区间", feats.index[0].date(), "~", feats.index[-1].date())
print("常规日", len(regular), "极端日", int((ext != '').sum()))
regular.head()

## 逐个 K 拟合并对比

`separation` 越大越分得开；`BIC` 只做横向参考；`min_share` / `avg_duration` / `stay_prob` 用来看状态是否"有用"。


In [ ]:
ks = (2, 3, 4, 5, 6)
res = {}
rows = []
for k in ks:
    r = k_study.fit_k(regular, k)
    res[k] = r
    prof = k_study.profile(r, regular)
    rows.append({"k": k, "sep": round(r["sep"]["min"], 3), "bic": round(r["bic"]),
                 "min_share": round(prof["share"].min(), 3),
                 "min_duration": round(prof["avg_duration"].min(), 1),
                 "min_stay": round(prof["stay_prob"].min(), 3)})

import pandas as pd
summary = pd.DataFrame(rows)
summary

## 每个 K 的状态画像

In [ ]:
for k in ks:
    print(f"===== K={k} =====")
    display = k_study.profile(res[k], regular).round(3)
    print(display.to_string(index=False))
    print()

## 市场分割图

价格曲线 + 按状态上色。看颜色块是否对应真实行情段落。


In [ ]:
from IPython.display import Image, display

for k in ks:
    p = k_study.plot_segments(res[k], regular, close.reindex(regular.index),
                              k_study.OUT / f"nb_segments_ashare_k{k}.png",
                              f"ashare  K={k}  separation={res[k]['sep']['min']:.2f}")
    display(Image(filename=str(p)))

## 读图要点

- 颜色块是否对应真实行情段落（2015 股灾、2018 熊市、2021 结构市、2024 反转）
- 有没有"只出现几天的碎块"——那是状态在追噪声
- 同一段行情里颜色是否稳定——不稳定说明特征没抓住那段行情
